In [86]:
#General packages
from datetime import datetime
from tqdm import tqdm
import os
import pandas as pd
import collections
import xml.etree.ElementTree as ET

#Extraction and Download packages
import urllib.request
import requests
import zipfile
import shutil
from bs4 import BeautifulSoup
import sqlite3
import gzip
import glob
import duckdb
import tarfile

### Set Path

In [87]:
data_path='/projects/ccnr/sebek.m/CPE_data/'

### Download and Preprocess PubChem

In [4]:
# Download PubChem bioactivities file
pubchem_dir = os.path.join(data_path, 'pubchem')
url = "https://ftp.ncbi.nlm.nih.gov/pubchem/Bioassay/Extras/bioactivities.tsv.gz"
response = requests.head(url)
total_size = int(response.headers.get('content-length', 0))

# Download with progress bar
response = requests.get(url, stream=True)
progress_bar = tqdm(total=total_size, unit='iB', unit_scale=True, desc="Downloading")

# Save data
temp_file = os.path.join(pubchem_dir, 'pc_bioactivities.tsv.gz')
with open(temp_file, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            progress_bar.update(len(chunk))
            f.write(chunk)
progress_bar.close()

print(f"Saved to {temp_file}")


Downloading:   0%|          | 0.00/2.98G [00:18<?, ?iB/s]

Downloading:   0%|          | 287k/2.98G [00:00<19:23, 2.56MiB/s]
Downloading:   0%|          | 1.43M/2.98G [00:00<06:37, 7.50MiB/s]
Downloading:   0%|          | 2.49M/2.98G [00:00<05:38, 8.80MiB/s]
Downloading:   0%|          | 3.60M/2.98G [00:00<05:11, 9.54MiB/s]
Downloading:   0%|          | 4.65M/2.98G [00:00<05:03, 9.81MiB/s]
Downloading:   0%|          | 5.64M/2.98G [00:00<05:10, 9.58MiB/s]
Downloading:   0%|          | 6.61M/2.98G [00:00<05:20, 9.27MiB/s]
Downloading:   0%|          | 7.54M/2.98G [00:00<05:31, 8.96MiB/s]
Downloading:   0%|          | 8.45M/2.98G [00:00<05:32, 8.94MiB/s]
Downloading:   0%|          | 9.35M/2.98G [00:01<05:33, 8.90MiB/s]
Downloading:   0%|          | 10.2M/2.98G [00:01<05:50, 8.48MiB/s]
Downloading:   0%|          | 11.1M/2.98G [00:01<06:01, 8.21MiB/s]
Downloading:   0%|          | 11.9M/2.98G [00:01<06:13, 7.94MiB/s]
Downloading:   0%|          | 12.7M/2.98G [00:01<06:22, 7.75MiB/s]
Dow

Saved to /projects/ccnr/sebek.m/CPE_data/pubchem/pc_bioactivities.tsv.gz


In [82]:
# Download PubChem bioactivities file
pubchem_dir = os.path.join(data_path, 'pubchem')
url = "https://ftp.ncbi.nlm.nih.gov/pubchem/Target/gene2info.gz"
response = requests.head(url)
total_size = int(response.headers.get('content-length', 0))

# Download with progress bar
response = requests.get(url, stream=True)
progress_bar = tqdm(total=total_size, unit='iB', unit_scale=True, desc="Downloading")

# Save data
temp_file = os.path.join(pubchem_dir, 'gene2info.gz')
with open(temp_file, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            progress_bar.update(len(chunk))
            f.write(chunk)
progress_bar.close()

print(f"Saved to {temp_file}")

Downloading: 100%|██████████| 30.2M/30.2M [00:00<00:00, 48.9MiB/s]

Saved to /projects/ccnr/sebek.m/CPE_data/pubchem/gene2info.gz


In [83]:
temp_file = os.path.join(pubchem_dir, 'gene2info.gz')
gene2info = pd.read_csv(temp_file, sep='\t', compression='gzip')
gene2info.to_csv(pubchem_dir+'/gene2info.csv')

In [ ]:
# Download CID-InChIKey file
url = "https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Extras/CID-InChI-Key.gz"
response = requests.head(url)
total_size = int(response.headers.get('content-length', 0))

# Download with progress bar
response = requests.get(url, stream=True)
progress_bar = tqdm(total=total_size, unit='iB', unit_scale=True, desc="Downloading")

# Save data
temp_file = os.path.join(pubchem_dir, 'CID-InChI-Key.gz')
with open(temp_file, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            progress_bar.update(len(chunk))
            f.write(chunk)
progress_bar.close()

print(f"Saved to {temp_file}")

In [88]:
# One-time setup script - Complete database creation
pubchem_dir = data_path+'pubchem'
bioact_file = os.path.join(pubchem_dir, 'pc_bioactivities.tsv.gz')
cid_inchikey_file = os.path.join(pubchem_dir, 'CID-InChI-Key.gz')
gene2info_file = os.path.join(pubchem_dir, 'gene2info.csv')
db_file = os.path.join(pubchem_dir, 'pubchem.duckdb')

con = duckdb.connect(db_file)

# Import bioactivities (~5-10 minutes)
print("Importing bioactivities...")
con.execute(f"""
    CREATE TABLE bioactivities AS 
    SELECT * FROM read_csv('{bioact_file}', delim='\t', compression='gzip', header=true)
""")

# Import CID-InChIKey with firstblock column
print("Importing CID-InChIKey mapping...")
con.execute(f"""
    CREATE TABLE cid_inchikey AS 
    SELECT 
        column0 AS CID,
        column1 AS InChI,
        column2 AS InChIKey,
        substr(column2, 1, 14) AS firstblock
    FROM read_csv('{cid_inchikey_file}', delim='\t', compression='gzip', 
                 header=false, auto_detect=false,
                 columns={{'column0': 'INTEGER', 'column1': 'VARCHAR', 'column2': 'VARCHAR'}})
""")

# Import gene2info
print("Importing gene2info...")
con.execute(f"""
    CREATE TABLE gene_info AS 
    SELECT * FROM read_csv('{gene2info_file}', header=true)
""")

# Create indices for fast lookups
print("Creating indices...")
con.execute("CREATE INDEX idx_bioact_cid ON bioactivities(CID)")
con.execute("CREATE INDEX idx_bioact_gene ON bioactivities(\"Gene ID\")")
con.execute("CREATE INDEX idx_cid_inchikey ON cid_inchikey(CID)")
con.execute("CREATE INDEX idx_firstblock ON cid_inchikey(firstblock)")
con.execute("CREATE INDEX idx_gene_id ON gene_info(GeneID)")
con.execute("CREATE INDEX idx_gene_tax ON gene_info(TaxonomyID)")

# Get stats
bioact_count = con.execute("SELECT COUNT(*) FROM bioactivities").fetchone()[0]
cid_count = con.execute("SELECT COUNT(*) FROM cid_inchikey").fetchone()[0]
gene_count = con.execute("SELECT COUNT(*) FROM gene_info").fetchone()[0]

con.close()

print(f"\nDatabase created: {db_file}")
print(f"Bioactivities: {bioact_count:,} rows")
print(f"CID-InChIKey: {cid_count:,} rows")
print(f"Gene info: {gene_count:,} rows")

Importing bioactivities...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Importing CID-InChIKey mapping...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Importing gene2info...
Creating indices...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Database created: /projects/ccnr/sebek.m/CPE_data/pubchem/pubchem.duckdb
Bioactivities: 299,401,168 rows
CID-InChIKey: 123,467,439 rows
Gene info: 1,645,898 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Found 11 filtered human interactions for CID 5280795


### Download and Preprocess BindingDB

In [3]:
# Extract BindingDB from manually downloaded file
with zipfile.ZipFile(data_path+'BindingDB_All_202512_tsv.zip', 'r') as zip_ref:
    zip_ref.extractall(data_path)  # Extract to current directory
print("Extraction complete")

Extraction complete


In [4]:
# Load extracted data, can take a while 6+ GB file
BDB_data=pd.read_csv(data_path+'BindingDB_All.tsv',sep='\t',low_memory=False, usecols=['Ligand SMILES','Ligand InChI','BindingDB MonomerID',
                    'Ligand InChI Key','BindingDB Ligand Name','Target Name','Target Source Organism According to Curator or DataSource', 
                    'Ki (nM)','IC50 (nM)','Kd (nM)','EC50 (nM)','pH','Temp (C)','Curation/DataSource',
                    'UniProt (SwissProt) Entry Name of Target Chain 1','UniProt (SwissProt) Primary ID of Target Chain 1'])

# Rename columns to match column names used in package
BDB_data=BDB_data.rename(columns={'Target Name':'Target Name Assigned by Curator or DataSource',
                                  'UniProt (SwissProt) Entry Name of Target Chain 1':'UniProt (SwissProt) Entry Name of Target Chain',
                                  'UniProt (SwissProt) Primary ID of Target Chain 1':'UniProt (SwissProt) Primary ID of Target Chain',
                                  'Ligand InChI Key':'inchikey'})

print(f"Retrieved {len(BDB_data)} compound-protein interactions")

In [6]:
BDB_data.keys()

Index(['Ligand SMILES', 'Ligand InChI', 'inchikey', 'BindingDB MonomerID',
       'BindingDB Ligand Name',
       'Target Name Assigned by Curator or DataSource',
       'Target Source Organism According to Curator or DataSource', 'Ki (nM)',
       'IC50 (nM)', 'Kd (nM)', 'EC50 (nM)', 'pH', 'Temp (C)',
       'Curation/DataSource', 'UniProt (SwissProt) Entry Name of Target Chain',
       'UniProt (SwissProt) Primary ID of Target Chain'],
      dtype='object')

In [7]:
BDB_data.to_csv(data_path+'BindingDB_Dec2025.csv')

### Download and Preprocess ChEMBL

In [3]:
# Download ChEMBL data using FTP
url = "https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/chembl_36_sqlite.tar.gz"
urllib.request.urlretrieve(url, "chembl.tar.gz")

# Extract from tar file
with tarfile.open(data_path+"chembl.tar.gz", "r:gz") as tar:
    tar.extractall(data_path)

# Connect to local sql database
conn = sqlite3.connect(data_path+"chembl_36/chembl_36_sqlite/chembl_36.db")

ContentTooShortError: <urlopen error retrieval incomplete: got only 546335725 out of 5611751319 bytes>

In [ ]:
# Query select interaction from sql
query = """
SELECT DISTINCT 
    md.chembl_id AS compound_chembl_id,
    md.pref_name AS compound_name,
    td.chembl_id AS target_chembl_id,
    td.pref_name AS target_name,
    td.target_type,
    td.organism,
    act.standard_type,
    act.standard_value,
    act.standard_units,
    act.pchembl_value,
    act.activity_comment,
    act.action_type,
    act.data_validity_comment,
    src.src_id AS source_id,
FROM molecule_dictionary md
JOIN compound_structures cs ON md.molregno = cs.molregno
JOIN activities act ON md.molregno = act.molregno
JOIN assays ass ON act.assay_id = ass.assay_id
JOIN target_dictionary td ON ass.tid = td.tid
LEFT JOIN target_components tc ON td.tid = tc.tid
LEFT JOIN component_sequences cs_comp ON tc.component_id = cs_comp.component_id
LEFT JOIN source src ON ass.src_id = src.src_id
WHERE act.standard_value IS NOT NULL
AND cs_comp.component_type = 'PROTEIN'
AND td.organism = 'Homo sapiens'
"""

# Store query in dataframe
chembl=pd.read_sql_query(query, conn)
print(f"Retrieved {len(chembl)} compound-protein interactions")

In [ ]:
# Rename columns to match column names in CPIExtract
chembl=chembl.rename(columns={'compound_chembl_id':'Molecule ChEMBL ID','compound_name':'Molecule Name','target_chembl_id':'Target ChEMBL ID',
                              'target_name':'Target Name','target_type':'Target Type','organism':'Target Organism','pchembl_value':'pChEMBL Value',
                              'standard_type':'Standard Type','standard_value':'Standard Value','standard_units':'Standard Units',
                              'activity_comment':'Comment','action_type':'Action Type','data_validity_comment':'Data Validity Comment',
                              'source_id':'Source ID'})

# Ensure only required columns are present
chembl = chembl[['Molecule ChEMBL ID','Molecule Name','Standard Type','Standard Value',
                 'Standard Units','pChEMBL Value','Data Validity Comment','Comment',
                 'Target ChEMBL ID','Target Name','Target Organism','Target Type','Source ID','Action Type']]

In [ ]:
chembl.to_csv(data_path+'ChEMBL_Dec2025.csv')

In [ ]:
chembl_data.keys()

### Download and Preprocess CTD

In [5]:
# Extract CTD from manually downloaded files
with gzip.open(data_path+'CTD_chem_gene_ixns.csv.gz', 'rb') as f_in:
    with open(data_path+'CTD_chem_gene_ixns.csv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
print("Chem-gene extraction complete")

#with gzip.open(data_path+'CTD_chemicals.csv.gz', 'rb') as f_in:
#    with open(data_path+'CTD_chemicals.csv', 'wb') as f_out:
#        shutil.copyfileobj(f_in, f_out)
#print("Chem-IDs extraction complete")

Chem-gene extraction complete
Chem-IDs extraction complete


In [15]:
# Load Extracted data files
rows_to_skip=list(range(27))+[28]
CTD_data=pd.read_csv(data_path+'CTD_chem_gene_ixns.csv',skiprows=rows_to_skip,header=0)
#rows_to_skip=list(range(27))+[28]
#CTD_chem=pd.read_csv(data_path+'CTD_chemicals.csv',skiprows=rows_to_skip,header=0)

# Rename column to match column name in CPIExtract
CTD_data=CTD_data.rename(columns={'# ChemicalName':'ChemicalName'})
#CTD_chem=CTD_chem.rename(columns={'# ChemicalName':'ChemicalName'})

# Merge Association data with chemical identifier data
#CTD_data=CTD_data.merge(CTD_chem,on='ChemicalName',how='left')

# Only keep required columns
CTD_data=CTD_data[['ChemicalName', 'ChemicalID', 'GeneSymbol', 'GeneID','GeneForms',
                   'Organism', 'OrganismID', 'Interaction','InteractionActions']]

print(f"Retrieved {len(CTD_data)} compound-protein interactions")

Retrieved 3019050 compound-protein interactions


In [16]:
CTD_data.keys()

Index(['ChemicalName', 'ChemicalID', 'GeneSymbol', 'GeneID', 'GeneForms',
       'Organism', 'OrganismID', 'Interaction', 'InteractionActions'],
      dtype='object')

In [17]:
CTD_data.to_csv(data_path+'CTD_Dec2025.csv')

### Download and Preprocess DrugBank

The DrugBank database requires an academic license to download, please refer to the DrugBank website for further instructions.

In [18]:
# Extract DrugBank
with zipfile.ZipFile(data_path+'drugbank_all_full_database.xml.zip', 'r') as zip_ref:
    zip_ref.extractall(data_path)  # Extract to current directory
print("Extraction complete")

tree = ET.parse(data_path+'full database.xml')
root = tree.getroot()
print("Loaded DB XML")

Extraction complete
Loaded DB XML


In [20]:
def extract_protein_info(protein_element, ns):
    """Extract protein ID, name, organism, and external IDs"""
    protein_data = {}
    
    # Get protein ID, name, and organism
    p_id = protein_element.find('db:id', ns)
    p_name = protein_element.find('db:name', ns)
    p_organism = protein_element.find('db:organism', ns)
    
    protein_data['protein_id'] = p_id.text if p_id is not None else ''
    protein_data['protein_name'] = p_name.text if p_name is not None else ''
    protein_data['organism'] = p_organism.text if p_organism is not None else ''
    
    # Extract external identifiers from polypeptide as separate fields
    external_ids = {}
    polypeptide = protein_element.find('db:polypeptide', ns)
    
    if polypeptide is not None:
        ext_ids = polypeptide.findall('.//db:external-identifiers/db:external-identifier', ns)
        for ext_id in ext_ids:
            resource = ext_id.find('db:resource', ns)
            identifier = ext_id.find('db:identifier', ns)
            if resource is not None and identifier is not None:
                # Store each external ID type separately
                external_ids[resource.text] = identifier.text
    
    protein_data['external_ids'] = external_ids
    
    return protein_data

def parse_drugbank_xml(xml_file):
    # Define namespace
    ns = {'db': 'http://www.drugbank.ca'}
    
    all_rows = []
    
    # Iterate through each drug entry
    for drug in root.findall('db:drug', ns):
        
        # Extract basic drug information
        primary_id = drug.find("db:drugbank-id[@primary='true']", ns)
        drugbank_id = primary_id.text if primary_id is not None else ''
        
        name_elem = drug.find('db:name', ns)
        drug_name = name_elem.text if name_elem is not None else ''
        
        # Extract synonyms
        synonyms = drug.findall('.//db:synonyms/db:synonym', ns)
        synonym_list = [syn.text for syn in synonyms if syn.text]
        synonyms_str = ' | '.join(synonym_list)
        
        # Extract PubChem CID from external identifiers
        pubchem_cid = ''
        external_ids = drug.findall('.//db:external-identifiers/db:external-identifier', ns)
        for ext_id in external_ids:
            resource = ext_id.find('db:resource', ns)
            identifier = ext_id.find('db:identifier', ns)
            if resource is not None and identifier is not None:
                if 'PubChem' in resource.text and 'Compound' in resource.text:
                    pubchem_cid = identifier.text
                    break
        
        # Extract InChIKey from calculated properties
        inchikey = ''
        calc_props = drug.findall('.//db:calculated-properties/db:property', ns)
        for prop in calc_props:
            kind = prop.find('db:kind', ns)
            value = prop.find('db:value', ns)
            if kind is not None and value is not None:
                if kind.text == 'InChIKey':
                    inchikey = value.text
                    break
        
        # Process targets
        targets = drug.findall('.//db:targets/db:target', ns)
        for target in targets:
            protein_info = extract_protein_info(target, ns)
            row = {
                'drugbank_id': drugbank_id,
                'name': drug_name,
                'synonyms': synonyms_str,
                'pubchem_cid': pubchem_cid,
                'inchikey': inchikey,
                'protein': protein_info['protein_id'],
                'protein_type': 'target',
                'protein_name': protein_info['protein_name'],
                'organism': protein_info['organism']
            }
            # Add external IDs as separate columns
            row.update(protein_info['external_ids'])
            all_rows.append(row)
        
        # Process enzymes
        enzymes = drug.findall('.//db:enzymes/db:enzyme', ns)
        for enzyme in enzymes:
            protein_info = extract_protein_info(enzyme, ns)
            row = {
                'drugbank_id': drugbank_id,
                'name': drug_name,
                'synonyms': synonyms_str,
                'pubchem_cid': pubchem_cid,
                'inchikey': inchikey,
                'protein': protein_info['protein_id'],
                'protein_type': 'enzyme',
                'protein_name': protein_info['protein_name'],
                'organism': protein_info['organism']
            }
            # Add external IDs as separate columns
            row.update(protein_info['external_ids'])
            all_rows.append(row)
        
        # Process carriers
        carriers = drug.findall('.//db:carriers/db:carrier', ns)
        for carrier in carriers:
            protein_info = extract_protein_info(carrier, ns)
            row = {
                'drugbank_id': drugbank_id,
                'name': drug_name,
                'synonyms': synonyms_str,
                'pubchem_cid': pubchem_cid,
                'inchikey': inchikey,
                'protein': protein_info['protein_id'],
                'protein_type': 'carrier',
                'protein_name': protein_info['protein_name'],
                'organism': protein_info['organism']
            }
            # Add external IDs as separate columns
            row.update(protein_info['external_ids'])
            all_rows.append(row)
        
        # Process transporters
        transporters = drug.findall('.//db:transporters/db:transporter', ns)
        for transporter in transporters:
            protein_info = extract_protein_info(transporter, ns)
            row = {
                'drugbank_id': drugbank_id,
                'name': drug_name,
                'synonyms': synonyms_str,
                'pubchem_cid': pubchem_cid,
                'inchikey': inchikey,
                'protein': protein_info['protein_id'],
                'protein_type': 'transporter',
                'protein_name': protein_info['protein_name'],
                'organism': protein_info['organism']
            }
            # Add external IDs as separate columns
            row.update(protein_info['external_ids'])
            all_rows.append(row)
    
    return all_rows


if __name__ == "__main__":
    drug_protein_data = parse_drugbank_xml(root)
    
    DB_data = pd.DataFrame(drug_protein_data)
    base_columns = ['drugbank_id', 'name', 'synonyms', 'pubchem_cid', 'inchikey',
                    'protein', 'protein_type', 'protein_name', 'organism']
    
    # Get all external ID columns (any column not in base_columns)
    if len(DB_data) > 0:
        external_id_columns = [col for col in DB_data.columns if col not in base_columns]
        external_id_columns = sorted(external_id_columns)  # Sort alphabetically
        
        # Final column order: base columns + external ID columns
        column_order = base_columns + external_id_columns
        DB_data = DB_data[column_order]
    print(f"External ID columns found: {len([col for col in DB_data.columns if col not in base_columns])}")
    
    # Save to CSV
    DB_data.to_csv(data_path+'DB_Dec2025.csv',index=False)
    
    print(f"Successfully extracted {len(DB_data)} drug-protein relationships")

External ID columns found: 9
Successfully extracted 37287 drug-protein relationships


In [22]:
# Rename columns to match CPIExtract input
DB_data=DB_data.rename(columns={'drugbank_id':'drugbank-id','HUGO Gene Nomenclature Committee (HGNC)':'HGNC'})

# Remove unnecessary columns
DB_data=DB_data[['drugbank-id', 'name', 'protein_name', 'protein_type', 'organism','HGNC']]

In [23]:
DB_data.keys()

Index(['drugbank-id', 'name', 'protein_name', 'protein_type', 'organism',
       'HGNC'],
      dtype='object')

In [24]:
DB_data.to_csv(data_path+'DB_Dec2025.csv')

### Download and Preprocess STITCH

In [31]:
# Extract STITCH from manual download of Human only data
with gzip.open(data_path+'9606.protein_chemical.links.detailed.v5.0.tsv.gz', 'rb') as f_in:
    with open(data_path+'9606.protein_chemical.links.detailed.v5.0.tsv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
print("Extraction complete")

Extraction complete


In [33]:
stch_data=pd.read_csv(data_path+'9606.protein_chemical.links.detailed.v5.0.tsv',sep='\t')
print(f"Retrieved {len(stch_data)} compound-protein interactions")

Retrieved 15473939 compound-protein interactions


In [36]:
stch_data.to_csv(data_path+'STITCH.csv')

In [47]:
stch_data.keys()

Index(['chemical', 'protein', 'experimental', 'prediction', 'database',
       'textmining', 'combined_score'],
      dtype='object')

### Download and Preprocess DTC

In [8]:
# Load extracted data, large file can take a while 
file_path=os.path.join(data_path, 'DTC.csv')
DTC_data=pd.read_csv(file_path,sep=',',usecols=['compound_id','target_id','gene_names','wildtype_or_mutant','mutation_info',
                                                'standard_type','standard_relation','standard_value','standard_units',
                                                'activity_comment','doc_type'])

/tmp/ipykernel_3172493/2252948939.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  DTC_data=pd.read_csv(file_path,sep=',',usecols=['compound_id','target_id','gene_names','wildtype_or_mutant','mutation_info',


In [9]:
# Remove entries without a compound ID
DTC_data=DTC_data.dropna(subset=['compound_id'])

print(f"Retrieved {len(DTC_data)} compound-protein interactions")

Retrieved 5846730 compound-protein interactions


In [10]:
DTC_data.to_csv(data_path+'DTC_Dec2025.csv')

In [13]:
DTC_data.keys()

Index(['compound_id', 'target_id', 'gene_names', 'wildtype_or_mutant',
       'mutation_info', 'standard_type', 'standard_relation', 'standard_value',
       'standard_units', 'activity_comment', 'doc_type'],
      dtype='object')

### Download and Preprocess DrugCentral

In [14]:
# Manually download file for processing
file_path=os.path.join(data_path, 'DrugCentral.csv')
DC_data=pd.read_csv(file_path, sep=',')

In [15]:
# Keep columns used in CPIExtract
DC_data=DC_data[['DRUG_NAME','STRUCT_ID', 'TARGET_NAME', 'TARGET_CLASS', 'ACCESSION',
       'GENE', 'SWISSPROT', 'ACT_VALUE', 'ACT_UNIT', 'ACT_TYPE', 'ACT_COMMENT','ACT_SOURCE',
       'RELATION', 'ACTION_TYPE', 'ORGANISM']]

print(f"Retrieved {len(DC_data)} compound-protein interactions")

Retrieved 23115 compound-protein interactions


In [16]:
DC_data.keys()

Index(['DRUG_NAME', 'STRUCT_ID', 'TARGET_NAME', 'TARGET_CLASS', 'ACCESSION',
       'GENE', 'SWISSPROT', 'ACT_VALUE', 'ACT_UNIT', 'ACT_TYPE', 'ACT_COMMENT',
       'ACT_SOURCE', 'RELATION', 'ACTION_TYPE', 'ORGANISM'],
      dtype='object')

In [17]:
DC_data.to_csv(data_path+'DrugCentral_Dec2025.csv')

### Download and Preprocess OTP

In [63]:
# Download only the Mechanisms of Action data from OTP
url = 'http://ftp.ebi.ac.uk/pub/databases/opentargets/platform/25.12/output/drug_mechanism_of_action/'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# Find all parquet files
parquet_links = [a['href'] for a in soup.find_all('a') if a['href'].endswith('.parquet')]
print(f"Found {len(parquet_links)} parquet files to download")

# Download each file
for filename in tqdm(parquet_links, desc="Downloading"):
    file_url = url + filename
    local_path = data_path+filename
    
    # Download file
    file_response = requests.get(file_url, stream=True)
    with open(local_path, 'wb') as f:
        for chunk in file_response.iter_content(chunk_size=8192):
            f.write(chunk)

# Load all files
parquet_files = glob.glob(data_path+'*.parquet')
print(f"Found {len(parquet_files)} local parquet files")
OTP_data = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)

print(f"Retrieved {len(OTP_data)} compound-protein interactions")

Found 1 parquet files to download


Downloading: 100%|██████████| 1/1 [00:07<00:00,  7.03s/it]


Found 1 local parquet files
Retrieved 6505 compound-protein interactions


In [65]:
# Keep columns used in CPIExtract
OTP_data=OTP_data[['actionType', 'mechanismOfAction', 'chemblIds', 'targetName',
                   'targetType', 'targets']]

In [66]:
OTP_data.to_csv(data_path+'OTP_Dec2025.csv')

In [68]:
OTP_data.keys()

Index(['actionType', 'mechanismOfAction', 'chemblIds', 'targetName',
       'targetType', 'targets'],
      dtype='object')